# SentraGrade — Supervised CNN Baseline (Module 3)

Trains an ImageNet-pretrained ResNet50 on the 5 in-distribution classes, for
each of the 6 leave-one-class-out folds. This is the reference point every
other backbone (SSL, ViT) and every OOD scoring method gets compared against.

**Before running:** upload `sentragrade_data.zip` (produced by
`01_preprocessing.py` on your Mac) to Google Drive, in the root of My Drive,
then run the cells below in order.

Runs on Colab (CUDA) or locally on your Mac (MPS) unchanged — device is
auto-detected.

In [1]:
import json, os, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

Using device: mps


## 1. Get the data

Colab: mounts Drive and unzips into local disk (`/content/data`) — much
faster to read during training than streaming thousands of small files off
Drive directly. Mac: point `DATA_ROOT` at wherever `01_preprocessing.py`
wrote its output (default `~/sentragrade_data`) and skip the unzip cell.

In [2]:
IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ZIP_PATH = "/content/drive/MyDrive/sentragrade_data.zip"  # <-- adjust if you put it elsewhere
    DATA_ROOT = Path("/content/data")
    DATA_ROOT.mkdir(exist_ok=True)

    if not (DATA_ROOT / "folds").exists():
        print("Unzipping dataset to local disk (one-time per session)...")
        with zipfile.ZipFile(ZIP_PATH) as zf:
            zf.extractall(DATA_ROOT)
        print("Done.")
    else:
        print("Already unzipped this session.")
else:
    candidates = [
        Path("~/sentragrade_data").expanduser(),
        Path.cwd(),
        Path.cwd().parent / "sentragrade_data",
    ]
    DATA_ROOT = next(
        (p for p in candidates if (p / "folds").exists() or (p / "manifest_full.csv").exists()),
        Path("~/sentragrade_data").expanduser(),
    )
    print(f"Using local dataset at: {DATA_ROOT}")

FOLD_DIR = DATA_ROOT / "folds"
assert FOLD_DIR.exists(), (
    f"Fold manifests not found at {FOLD_DIR}. "
    "Expected a folder like ~/sentragrade_data created by preprocessing.py."
)
print(f"DATA_ROOT = {DATA_ROOT}")

Using local dataset at: /Users/riteeshtm/sentragrade_data
DATA_ROOT = /Users/riteeshtm/sentragrade_data


## 2. Dataset and transforms

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ProduceDataset(Dataset):
    """Reads a fold manifest CSV. label is None for the OOD (held-out class)
    split, since those images have no valid label among the 5 known classes."""

    def __init__(self, csv_path, label_map, transform, data_root):
        self.df = pd.read_csv(csv_path)
        self.label_map = label_map
        self.transform = transform
        self.data_root = Path(data_root)

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, row):
        # resized_path was written as an absolute Mac path by preprocessing;
        # on Colab we only kept class_name/filename inside the zip, so
        # re-root it under DATA_ROOT/resized.
        return self.data_root / "resized" / row.class_name / row.filename

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_path(row)
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        label = self.label_map.get(row.class_name, -1)  # -1 for OOD class
        return img, label, str(img_path)

## 3. Model — ResNet50, partial fine-tune, exposes both logits and pooled features

In [4]:
class ResNetClassifier(nn.Module):
    """Wraps torchvision resnet50. freeze_until controls how much of the
    backbone is frozen: 'layer2' (default) freezes conv1/bn1/layer1/layer2
    and fine-tunes layer3, layer4, fc — a reasonable balance of speed and
    accuracy for a dataset this size. Returns (logits, 2048-d pooled
    features) — the features feed the prototype-distance OOD score later,
    and this same checkpoint is what Grad-CAM hooks into downstream."""

    def __init__(self, num_classes, freeze_until="layer2"):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = backbone.avgpool
        self.fc = nn.Linear(backbone.fc.in_features, num_classes)

        freeze_order = ["stem", "layer1", "layer2", "layer3", "layer4"]
        freeze_idx = freeze_order.index(freeze_until)
        for name in freeze_order[: freeze_idx + 1]:
            for p in getattr(self, name).parameters():
                p.requires_grad = False

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        feats = torch.flatten(x, 1)
        logits = self.fc(feats)
        return logits, feats

    def trainable_param_groups(self, backbone_lr=1e-4, head_lr=1e-3):
        backbone_params = [p for n, p in self.named_parameters() if p.requires_grad and not n.startswith("fc")]
        head_params = [p for p in self.fc.parameters() if p.requires_grad]
        return [
            {"params": backbone_params, "lr": backbone_lr},
            {"params": head_params, "lr": head_lr},
        ]

## 4. Train one fold

In [5]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels, _ in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits, _ = model(imgs)
            loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            n += imgs.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def collect_logits_features(model, loader):
    model.eval()
    all_logits, all_feats, all_labels, all_paths = [], [], [], []
    for imgs, labels, paths in loader:
        imgs = imgs.to(DEVICE)
        logits, feats = model(imgs)
        all_logits.append(logits.cpu())
        all_feats.append(feats.cpu())
        all_labels.append(labels)
        all_paths.extend(paths)
    return {
        "logits": torch.cat(all_logits),
        "features": torch.cat(all_feats),
        "labels": torch.cat(all_labels),
        "paths": all_paths,
    }


def train_one_fold(fold_idx, epochs=25, batch_size=32, patience=6, num_workers=0):
    # On macOS + Jupyter/Notebook, DataLoader with multiprocessing workers can
    # fail to pickle the custom Dataset class. Keep workers at 0 here unless you
    # run from a normal Python script with a proper __main__ guard.
    print(f"\n{'='*60}\nFold {fold_idx}\n{'='*60}")

    with open(FOLD_DIR / f"fold{fold_idx}_label_map.json") as f:
        label_map = json.load(f)
    num_classes = len(label_map)

    train_ds = ProduceDataset(FOLD_DIR / f"fold{fold_idx}_train.csv", label_map, train_tf, DATA_ROOT)
    val_ds = ProduceDataset(FOLD_DIR / f"fold{fold_idx}_val.csv", label_map, eval_tf, DATA_ROOT)
    ood_ds = ProduceDataset(FOLD_DIR / f"fold{fold_idx}_ood.csv", label_map, eval_tf, DATA_ROOT)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    ood_loader = DataLoader(ood_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    train_labels = train_ds.df["class_name"].map(label_map).values
    class_weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=train_labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model = ResNetClassifier(num_classes, freeze_until="layer2").to(DEVICE)
    optimizer = torch.optim.AdamW(model.trainable_param_groups(), weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_acc, best_state, epochs_no_improve = 0.0, None, 0
    for epoch in range(epochs):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()
        dt = time.time() - t0
        print(f"epoch {epoch+1:2d}/{epochs}  train_loss={train_loss:.3f} train_acc={train_acc:.3f}  "
              f"val_loss={val_loss:.3f} val_acc={val_acc:.3f}  ({dt:.1f}s)")

        if val_acc > best_val_acc:
            best_val_acc, best_state, epochs_no_improve = val_acc, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1} (best val_acc={best_val_acc:.3f})")
                break

    model.load_state_dict(best_state)

    val_results = collect_logits_features(model, val_loader)
    ood_results = collect_logits_features(model, ood_loader)

    ckpt_dir = DATA_ROOT / "checkpoints" / "supervised_cnn"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "fold_idx": fold_idx,  # see folds/fold_summary.csv for the held-out class name
            "label_map": label_map,
            "model_state_dict": best_state,
            "best_val_acc": best_val_acc,
            "val": val_results,
            "ood": ood_results,
        },
        ckpt_dir / f"fold{fold_idx}.pt",
    )
    print(f"Saved checkpoint + logits/features -> {ckpt_dir / f'fold{fold_idx}.pt'}")
    return best_val_acc

## 5. Run all 6 folds

This is the expensive part. If a Colab session drops mid-way, just re-run
this cell — folds whose checkpoint file already exists are skipped, so you
only pay for the folds that haven't finished.

In [6]:
fold_summary = pd.read_csv(FOLD_DIR / "fold_summary.csv")
results = []

for fold_idx in fold_summary["fold"]:
    ckpt_path = DATA_ROOT / "checkpoints" / "supervised_cnn" / f"fold{fold_idx}.pt"
    if ckpt_path.exists():
        print(f"Fold {fold_idx} already done, skipping (delete {ckpt_path} to redo).")
        saved = torch.load(ckpt_path, map_location="cpu")
        val_acc = saved["best_val_acc"]
    else:
        val_acc = train_one_fold(fold_idx)
    held_out = fold_summary.loc[fold_summary.fold == fold_idx, "held_out_class"].item()
    results.append({"fold": fold_idx, "held_out_class": held_out, "best_val_acc": val_acc})

results_df = pd.DataFrame(results)
print("\n=== Supervised CNN baseline — per-fold in-distribution val accuracy ===")
print(results_df)
results_df.to_csv(DATA_ROOT / "checkpoints" / "supervised_cnn" / "summary.csv", index=False)


Fold 0
epoch  1/25  train_loss=0.124 train_acc=0.983  val_loss=0.000 val_acc=1.000  (59.2s)
epoch  2/25  train_loss=0.007 train_acc=0.999  val_loss=0.000 val_acc=1.000  (51.8s)
epoch  3/25  train_loss=0.001 train_acc=1.000  val_loss=0.000 val_acc=1.000  (50.9s)
epoch  4/25  train_loss=0.000 train_acc=1.000  val_loss=0.000 val_acc=1.000  (50.5s)
epoch  5/25  train_loss=0.005 train_acc=0.999  val_loss=0.000 val_acc=1.000  (52.1s)
epoch  6/25  train_loss=0.002 train_acc=0.999  val_loss=0.007 val_acc=0.998  (51.6s)
epoch  7/25  train_loss=0.002 train_acc=0.999  val_loss=0.000 val_acc=1.000  (52.8s)
Early stopping at epoch 7 (best val_acc=1.000)
Saved checkpoint + logits/features -> /Users/riteeshtm/sentragrade_data/checkpoints/supervised_cnn/fold0.pt

Fold 1
epoch  1/25  train_loss=0.138 train_acc=0.969  val_loss=0.002 val_acc=1.000  (50.4s)
epoch  2/25  train_loss=0.008 train_acc=0.998  val_loss=0.008 val_acc=0.996  (49.3s)
epoch  3/25  train_loss=0.001 train_acc=1.000  val_loss=0.001 va

## Notes for next steps

- `val_acc` above is plain 5-class classification accuracy on the
  in-distribution validation set — a sanity check that the baseline
  actually learned something, **not** the OOD detection metric. AUROC /
  FPR@95 comes later once you compute energy score, MSP, and prototype
  distance from the saved `val`/`ood` logits and features in each
  checkpoint (module 7–9 in the report).
- Each `checkpoints/supervised_cnn/fold{i}.pt` contains everything the
  later modules need: the trained weights (for Grad-CAM), and cached
  logits + 2048-d features for both the in-distribution val set and the
  true held-out OOD class — so you never have to re-run inference to get
  those.
- If you're running this on your Mac (MPS) instead of Colab, drop
  `batch_size` to 16 in `train_one_fold(...)` if you hit memory pressure,
  and expect it to be noticeably slower per epoch than Colab's T4.